# Codex final: PPO MLP vs DMLPA ablation

This notebook is the architecture ablation runner. It is not the main scientific ladder. The main claim should come from the static/adaptive ladder; this notebook only asks whether a DMLPA-style feature extractor adds value under the same declared action contract.

Default is a short debug run. For a serious Colab run, set `RUN_PROFILE = "serious"`.


In [ ]:
# =====================
# 0) Run config
# =====================

RUN_PROFILE = "debug"  # debug or serious
CONTRACT_PROFILE = "scientific_thesis_factorized"  # scientific_thesis_factorized or legacy_box18

GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "main"
REPO_DIR_COLAB = "/content/scresia"
REPO_DIR_KAGGLE = "/kaggle/working/scresia"

USE_GOOGLE_DRIVE_OUTPUT = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/scresia_results/codex_final_dmlpa_ablation"
LOCAL_OUTPUT_DIR = "/content/codex_final_dmlpa_ablation"
KAGGLE_OUTPUT_DIR = "/kaggle/working/codex_final_dmlpa_ablation"

RUNTIME_PIP_PACKAGES = [
    "simpy>=4.1",
    "numpy>=1.26",
    "gymnasium>=0.29",
    "stable-baselines3>=2.3",
    "torch>=2.1",
    "einops>=0.7",
    "pandas>=2.2",
    "scipy>=1.13",
]

BASE_SEED = 42
REWARD_MODE = "ReT_cd_v1"
RISK_LEVEL = "increased"
STOCHASTIC_PT = True
STEP_SIZE_HOURS = 168.0
FRAME_STACK = 30
FEATURES_DIM = 120

if RUN_PROFILE == "debug":
    TRAIN_SEEDS = [101]
    TOTAL_TIMESTEPS = 512
    MAX_STEPS = 8
    EVAL_EPISODES = 1
    N_STEPS = 64
    BATCH_SIZE = 32
    N_EPOCHS = 2
    POLICY_LABELS = ["ppo_mlp_baseline", "ppo_friend_dmlpa_original"]
elif RUN_PROFILE == "serious":
    TRAIN_SEEDS = [101, 202, 303]
    TOTAL_TIMESTEPS = 200_000
    MAX_STEPS = 260
    EVAL_EPISODES = 5
    N_STEPS = 1024
    BATCH_SIZE = 256
    N_EPOCHS = 10
    POLICY_LABELS = ["ppo_mlp_baseline", "ppo_friend_dmlpa_original", "ppo_friend_dmlpa_positional"]
else:
    raise ValueError("RUN_PROFILE must be debug or serious")

print({
    "RUN_PROFILE": RUN_PROFILE,
    "CONTRACT_PROFILE": CONTRACT_PROFILE,
    "POLICY_LABELS": POLICY_LABELS,
    "TRAIN_SEEDS": TRAIN_SEEDS,
    "TOTAL_TIMESTEPS": TOTAL_TIMESTEPS,
    "MAX_STEPS": MAX_STEPS,
})


In [ ]:
# ==========================================
# 1) Portable setup: Colab, Kaggle, or local
# ==========================================

from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").exists()
if not (IN_COLAB or IN_KAGGLE):
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {cmd}")
    return result

if IN_COLAB and USE_GOOGLE_DRIVE_OUTPUT:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True, timeout_ms=120_000)
        drive_mounted = True
    except Exception as exc:
        print("WARNING: Drive mount failed; using /content outputs.", repr(exc))
        drive_mounted = False
else:
    drive_mounted = False

if IN_COLAB or IN_KAGGLE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PIP_PACKAGES])

if IN_COLAB:
    repo_root = Path(REPO_DIR_COLAB)
    shutil.rmtree(repo_root, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(repo_root)])
    output_root = Path(DRIVE_OUTPUT_DIR if drive_mounted else LOCAL_OUTPUT_DIR)
elif IN_KAGGLE:
    repo_root = Path(REPO_DIR_KAGGLE)
    shutil.rmtree(repo_root, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(repo_root)])
    output_root = Path(KAGGLE_OUTPUT_DIR)
else:
    cwd = Path.cwd().resolve()
    repo_root = cwd if (cwd / "supply_chain").exists() else cwd.parent
    output_root = repo_root / "outputs" / "codex_final_dmlpa_ablation"

sys.path.insert(0, str(repo_root.parent))
sys.path.insert(0, str(repo_root))
output_root.mkdir(parents=True, exist_ok=True)

git_hash = run_cmd(["git", "rev-parse", "HEAD"], cwd=repo_root).stdout.strip()
print("repo_root:", repo_root)
print("output_root:", output_root)
print("git_hash:", git_hash)


In [ ]:
# ===========================
# 2) Imports and env contract
# ===========================

import gymnasium as gym
import numpy as np
import pandas as pd
import torch
from scipy import stats
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecNormalize

from supply_chain.external_env_interface import get_episode_terminal_metrics, make_dkana_thesis_faithful_env

random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)

CONTRACT_KWARGS = {
    "scientific_thesis_factorized": dict(
        observation_version="v5",
        observation_mode="env_sdm_history_reward",
        action_space_mode="thesis_factorized",
        inventory_period_mode="thesis_strict",
        learn_initial_decision=True,
    ),
    "legacy_box18": dict(
        observation_mode="decision_reward",
        action_space_mode="onehot_18d",
        inventory_period_mode="thesis_strict",
        learn_initial_decision=False,
    ),
}
ENV_KWARGS = dict(
    reward_mode=REWARD_MODE,
    risk_level=RISK_LEVEL,
    stochastic_pt=STOCHASTIC_PT,
    max_steps=MAX_STEPS,
    step_size_hours=STEP_SIZE_HOURS,
    **CONTRACT_KWARGS[CONTRACT_PROFILE],
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("ENV_KWARGS:", ENV_KWARGS)


In [ ]:
# ===============================
# 3) Env factories and smoke check
# ===============================


def make_raw_env(seed: int):
    env = make_dkana_thesis_faithful_env(**ENV_KWARGS)
    env.reset(seed=seed)
    return env


def make_single_env(seed: int):
    def _init():
        env = make_dkana_thesis_faithful_env(**ENV_KWARGS)
        env.reset(seed=seed)
        return Monitor(env)
    return _init


def make_vec_env(seed: int) -> VecNormalize:
    vec = DummyVecEnv([make_single_env(seed)])
    vec = VecFrameStack(vec, n_stack=FRAME_STACK)
    vec = VecNormalize(vec, norm_obs=True, norm_reward=True)
    return vec

smoke_env = make_raw_env(BASE_SEED)
obs, info = smoke_env.reset(seed=BASE_SEED)
print("single obs shape:", obs.shape)
print("action space:", smoke_env.action_space)
print("action_space_mode:", info.get("action_space_mode"))
print("inventory_period_mode:", info.get("inventory_period_mode"))
if CONTRACT_PROFILE == "scientific_thesis_factorized":
    assert smoke_env.action_space.nvec.tolist() == [6, 3]
else:
    assert smoke_env.action_space.shape == (18,)
smoke_env.close()

probe_vec = make_vec_env(BASE_SEED)
probe_obs = probe_vec.reset()
print("stacked obs shape:", probe_obs.shape)
print("vector action space:", probe_vec.action_space)
assert probe_vec.observation_space.shape[0] % FRAME_STACK == 0
probe_vec.close()


In [ ]:
# ============================
# 4) DMLPA feature extractors
# ============================

class FriendDMLPAOriginal(BaseFeaturesExtractor):
    """Original friend DMLPA: Linear -> GELU -> Linear -> TransformerEncoder -> last token."""

    def __init__(self, observation_space: gym.spaces.Box, factor: int = FRAME_STACK, features_dim: int = FEATURES_DIM):
        super().__init__(observation_space, features_dim)
        flat_dim = int(np.prod(observation_space.shape))
        if flat_dim % factor != 0:
            raise ValueError(f"Observation dim {flat_dim} is not divisible by factor={factor}")
        self.factor = int(factor)
        self.obs_dimension = flat_dim // self.factor
        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.obs_dimension, 100),
            torch.nn.GELU(),
            torch.nn.Linear(100, features_dim),
        )
        self.attlayer = torch.nn.TransformerEncoderLayer(d_model=features_dim, nhead=12, batch_first=True)
        self.accumulated = torch.nn.TransformerEncoder(self.attlayer, num_layers=4)

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        x = observations.reshape(observations.shape[0], self.factor, self.obs_dimension)
        x = self.latent_rw(x)
        x = self.accumulated(x)
        return x[:, -1, :]


class FriendDMLPAPositional(BaseFeaturesExtractor):
    """Positional ablation from the new notebook: sinusoidal PE + LayerNorm."""

    def __init__(self, observation_space: gym.spaces.Box, factor: int = FRAME_STACK, features_dim: int = FEATURES_DIM):
        super().__init__(observation_space, features_dim)
        flat_dim = int(np.prod(observation_space.shape))
        if flat_dim % factor != 0:
            raise ValueError(f"Observation dim {flat_dim} is not divisible by factor={factor}")
        self.factor = int(factor)
        self.obs_dimension = flat_dim // self.factor
        self.latent_rw = torch.nn.Sequential(
            torch.nn.Linear(self.obs_dimension, 100),
            torch.nn.GELU(),
            torch.nn.Linear(100, features_dim),
        )
        self.pre_norm = torch.nn.LayerNorm(features_dim)
        self.attlayer = torch.nn.TransformerEncoderLayer(d_model=features_dim, nhead=12, batch_first=True)
        self.accumulated = torch.nn.TransformerEncoder(self.attlayer, num_layers=4)
        self.register_buffer("pos_encoding", self._build_sinusoidal_pe(self.factor, features_dim))

    @staticmethod
    def _build_sinusoidal_pe(seq_len: int, d_model: int) -> torch.Tensor:
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        x = observations.reshape(observations.shape[0], self.factor, self.obs_dimension)
        x = self.latent_rw(x)
        x = self.pre_norm(x + self.pos_encoding.to(dtype=x.dtype))
        x = self.accumulated(x)
        return x[:, -1, :]

POLICY_KWARGS = {
    "ppo_mlp_baseline": dict(net_arch=dict(pi=[256, 256], vf=[256, 256])),
    "ppo_friend_dmlpa_original": dict(
        features_extractor_class=FriendDMLPAOriginal,
        features_extractor_kwargs=dict(factor=FRAME_STACK, features_dim=FEATURES_DIM),
        net_arch=dict(pi=[128], vf=[128]),
    ),
    "ppo_friend_dmlpa_positional": dict(
        features_extractor_class=FriendDMLPAPositional,
        features_extractor_kwargs=dict(factor=FRAME_STACK, features_dim=FEATURES_DIM),
        net_arch=dict(pi=[128], vf=[128]),
    ),
}


In [ ]:
# ========================
# 5) Train selected models
# ========================

@dataclass
class TrainedPolicy:
    label: str
    train_seed: int
    model: PPO
    vec_env: VecNormalize
    elapsed_seconds: float
    model_path: Path
    norm_path: Path


def build_model(label: str, vec_env: VecNormalize, seed: int) -> PPO:
    return PPO(
        "MlpPolicy",
        vec_env,
        policy_kwargs=POLICY_KWARGS[label],
        learning_rate=3e-4,
        n_steps=N_STEPS,
        batch_size=BATCH_SIZE,
        n_epochs=N_EPOCHS,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.20,
        seed=seed,
        device="auto",
        verbose=1,
    )

trained_policies = []
training_rows = []
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
for label in POLICY_LABELS:
    for seed in TRAIN_SEEDS:
        policy_dir = output_root / label / f"seed_{seed}_{run_stamp}"
        policy_dir.mkdir(parents=True, exist_ok=True)
        vec_env = make_vec_env(seed)
        model = build_model(label, vec_env, seed)
        start = time.time()
        model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=False)
        elapsed = time.time() - start
        model_path = policy_dir / "model.zip"
        norm_path = policy_dir / "vecnormalize.pkl"
        model.save(model_path)
        vec_env.save(norm_path)
        trained = TrainedPolicy(label, seed, model, vec_env, elapsed, model_path, norm_path)
        trained_policies.append(trained)
        training_rows.append({"policy": label, "seed": seed, "elapsed_seconds": elapsed, "model_path": str(model_path), "norm_path": str(norm_path)})
        pd.DataFrame(training_rows).to_csv(output_root / "training_runs.csv", index=False)

display(pd.DataFrame(training_rows))


In [ ]:
# ==========================================
# 6) Manual deterministic evaluation
# ==========================================


def normalize_stacked_obs(stacked: np.ndarray, vecnorm: VecNormalize) -> np.ndarray:
    normalized = (stacked - vecnorm.obs_rms.mean) / np.sqrt(vecnorm.obs_rms.var + vecnorm.epsilon)
    return np.clip(normalized, -vecnorm.clip_obs, vecnorm.clip_obs).astype(np.float32)


def evaluate_policy(trained: TrainedPolicy) -> pd.DataFrame:
    rows = []
    trained.vec_env.training = False
    trained.vec_env.norm_reward = False
    for episode in range(EVAL_EPISODES):
        eval_seed = 1_000_000 + trained.train_seed * 100 + episode
        env = make_raw_env(eval_seed)
        obs, info = env.reset(seed=eval_seed)
        frames = deque([np.zeros_like(obs, dtype=np.float32) for _ in range(FRAME_STACK - 1)] + [obs.astype(np.float32)], maxlen=FRAME_STACK)
        terminated = truncated = False
        total_reward = 0.0
        steps = 0
        while not (terminated or truncated):
            stacked = np.concatenate(list(frames), axis=0).reshape(1, -1)
            model_obs = normalize_stacked_obs(stacked, trained.vec_env)
            action, _ = trained.model.predict(model_obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action[0])
            frames.append(obs.astype(np.float32))
            total_reward += float(reward)
            if info.get("action_phase") == "weekly_decision":
                steps += 1
        terminal = get_episode_terminal_metrics(env)
        rows.append({
            "policy": trained.label,
            "train_seed": trained.train_seed,
            "episode": episode,
            "eval_seed": eval_seed,
            "steps": steps,
            "total_reward": total_reward,
            **terminal,
        })
        env.close()
    return pd.DataFrame(rows)

parts = []
for trained in trained_policies:
    print("Evaluating", trained.label, trained.train_seed)
    part = evaluate_policy(trained)
    parts.append(part)
    pd.concat(parts, ignore_index=True).to_csv(output_root / "evaluation_episodes.csv", index=False)

eval_df = pd.concat(parts, ignore_index=True)
display(eval_df)


In [ ]:
# =======================================
# 7) Aggregates and paired comparisons
# =======================================

PRIMARY_METRICS = ["total_reward", "order_level_ret_mean", "fill_rate_order_level", "backorder_rate_order_level"]
seed_summary = eval_df.groupby(["policy", "train_seed"], as_index=False)[PRIMARY_METRICS].mean()
policy_summary = seed_summary.groupby("policy", as_index=False).agg({m: ["mean", "std", "count"] for m in PRIMARY_METRICS})
policy_summary.columns = ["_".join([str(x) for x in col if x]) for col in policy_summary.columns.to_flat_index()]
seed_summary.to_csv(output_root / "seed_summary.csv", index=False)
policy_summary.to_csv(output_root / "policy_summary.csv", index=False)

display(seed_summary)
display(policy_summary)

comparison_rows = []
if "ppo_mlp_baseline" in set(seed_summary["policy"]):
    for candidate in [p for p in POLICY_LABELS if p != "ppo_mlp_baseline"]:
        for metric in PRIMARY_METRICS:
            pivot = seed_summary.pivot(index="train_seed", columns="policy", values=metric).dropna()
            if candidate not in pivot.columns or "ppo_mlp_baseline" not in pivot.columns:
                continue
            diff = pivot[candidate].to_numpy() - pivot["ppo_mlp_baseline"].to_numpy()
            t_p = stats.ttest_rel(pivot[candidate], pivot["ppo_mlp_baseline"]).pvalue if len(diff) >= 2 else np.nan
            comparison_rows.append({
                "candidate_policy": candidate,
                "reference_policy": "ppo_mlp_baseline",
                "metric": metric,
                "n_paired_train_seeds": len(diff),
                "candidate_mean": float(pivot[candidate].mean()),
                "reference_mean": float(pivot["ppo_mlp_baseline"].mean()),
                "paired_diff_candidate_minus_reference": float(np.mean(diff)),
                "paired_t_p": float(t_p) if np.isfinite(t_p) else np.nan,
            })
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(output_root / "paired_comparison.csv", index=False)
display(comparison_df)


In [ ]:
# =============================
# 8) Save run manifest
# =============================

manifest = {
    "notebook": "codex_final_dmlpa_ablation_colab",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "git_hash": git_hash,
    "run_profile": RUN_PROFILE,
    "contract_profile": CONTRACT_PROFILE,
    "policy_labels": POLICY_LABELS,
    "train_seeds": TRAIN_SEEDS,
    "total_timesteps": TOTAL_TIMESTEPS,
    "max_steps": MAX_STEPS,
    "eval_episodes": EVAL_EPISODES,
    "frame_stack": FRAME_STACK,
    "env_kwargs": ENV_KWARGS,
    "claim_boundary": "DMLPA is an architecture ablation under a declared action contract, not the main thesis-relaxation result.",
}
(output_root / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print("output_root:", output_root)
